In [1]:
from part1.bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tt = tokenizer.load("bpe_tokenizer.json")
PAD_ID = len(tt.vocab)
MASK_ID = PAD_ID +1 
EOS_ID = MASK_ID + 1
CANVAS_LENGTH = 350

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

class Attention(nn.Module):
    def __init__(self, d_model, attn_dim):
        super().__init__()
        self.d_model = d_model
        self.attn_dim = attn_dim
        self.scaling_factor = attn_dim ** 0.5
        self.W_q = nn.Linear(d_model, attn_dim, bias=False)
        self.W_k = nn.Linear(d_model, attn_dim, bias=False)
        self.W_v = nn.Linear(d_model, attn_dim, bias=False)

    def forward(self, x, mask=None):
        queries = self.W_q(x)
        keys = self.W_k(x)
        scores = queries @ keys.transpose(-2, -1) / self.scaling_factor

        if mask is not None:
            # mask: (..., S) -> (..., 1, S) broadcasts correctly over query dim
            scores = scores.masked_fill(mask.unsqueeze(-2) == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        values = self.W_v(x)
        return attn_weights @ values

class MultiheadAttention(nn.Module):
    def __init__(self, d_model, attn_dim, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([Attention(d_model, attn_dim) for _ in range(num_heads)])
        self.output_projection = nn.Linear(attn_dim * num_heads, d_model)

    def forward(self, x, mask=None):
        head_outputs = [head(x, mask) for head in self.heads]
        return self.output_projection(torch.cat(head_outputs, dim=-1))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, attn_dim, num_heads, ff_dim=1024, dropout=0.2):
        super().__init__()
        self.attn = MultiheadAttention(d_model, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.norm1(x + self.drop1(self.attn(x, mask)))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class MaskedLM(nn.Module):
    def __init__(self, vocab_size, max_seq_len=256, d_model=256, num_layers=3, dropout=0.2):
        super().__init__()
        # vocab_size+3 covers all IDs: regular vocab + PAD + MASK + EOS
        self.embedding = nn.Embedding(vocab_size + 3, d_model, padding_idx=PAD_ID)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_seq_len)
        self.embed_drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, 32, 8, ff_dim=512, dropout=dropout)
            for _ in range(num_layers)
        ])
        # vocab_size+1 outputs: regular vocab (0..vocab_size-1) + EOS (vocab_size)
        self.lm_head = nn.Linear(d_model, vocab_size + 1)

    def forward(self, x, attn_masks=None):
        out = self.embed_drop(self.pos_encoding(self.embedding(x)))
        for layer in self.layers:
            out = layer(out, attn_masks)
        return self.lm_head(out)

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
### [TESTING]

original_sequence = "The quick brown fox jumps over the lazy dog"
tokens = tt.encode(original_sequence)
print("Tokens:", tokens)

Tokens: [349, 2020, 1103, 643, 276, 3824, 479, 434, 1750, 685, 260, 312, 1615, 121, 2873]


In [ ]:
### [TESTING]

L = 50
X = 30
res = torch.bernoulli(torch.full((L,), X / 100))
res

tensor([1., 0., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
        0., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 1., 0.,
        1., 0., 1., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0.])

In [ ]:
### [TESTING]

import torch.nn.functional as F
def preproc(sequence, max_length):
    tokens = torch.tensor(sequence)
    
    # padding
    pred_masks = torch.bernoulli(torch.full((len(tokens),), X / 100)).bool()
    corrupted = torch.masked_fill(tokens, pred_masks, MASK_ID)

    padded = F.pad(tokens, (0, max_length - len(tokens)), value=len(tt.vocab))
    corrupted = F.pad(corrupted, (0, max_length - len(corrupted)), value=PAD_ID)
    pred_masks = F.pad(pred_masks, (0, max_length - len(pred_masks)), value=0).bool()
    attn_masks = (padded != len(tt.vocab)).long()


    return padded, corrupted, attn_masks, pred_masks
    

In [ ]:
### [TESTING]

original_seq, corrupted_seq, attn_masks, pred_masks = preproc(tokens, 50)

## Training


In [4]:
model = MaskedLM(vocab_size=len(tt.vocab), max_seq_len=CANVAS_LENGTH).to(device)

In [5]:
print(corrupted_seq.shape)
print(attn_masks.shape)
lm_logits = model(corrupted_seq, attn_masks) # 1, seq_len, vocab_size
print("LM logits shape:", lm_logits.shape)
loss = 0.0000001
logits = lm_logits[0]
if pred_masks.any():
    print(logits[pred_masks].shape)
    print(original_seq[pred_masks].shape)
    loss = F.cross_entropy(logits[pred_masks], original_seq[pred_masks])
else:
    loss = torch.tensor(0.0, device=logits.device)

loss.item()

NameError: name 'corrupted_seq' is not defined

## Data magic

In [6]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TrainDataset(Dataset):
    def __init__(self, sequences, L=CANVAS_LENGTH, PAD_ID=PAD_ID):
        self.sequences = torch.full((len(sequences), L), PAD_ID, dtype=torch.long)

        for i, seq in enumerate(sequences):
            length = min(len(seq), L)
            self.sequences[i, :length] = torch.tensor(seq[:length], dtype=torch.long)

        # All L positions are in-canvas: semantic tokens, EOS, and learned PAD tail
        # all get attn_mask=1. Only sequences longer than L (impossible here) would
        # need batch-only padding with attn_mask=0.
        self.attn_masks = torch.ones(len(sequences), L, dtype=torch.bool)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.attn_masks[idx]

In [27]:
samples: list[str] = []

buckets_bounds = np.arange(0, 500 + 50, 50)
buckets = {b: [] for b in buckets_bounds}

with open("challenge-data/train.txt", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t", maxsplit=1)
        if len(parts) < 2:
            continue
        input_string = parts[1]
        res = tt.encode(input_string) + [EOS_ID]  # Ensure the string can be encoded

        l = len(res)
        for b in buckets_bounds:
            if l <= b:
                buckets[b].append(res)
                break

for k, v in buckets.items():
    if k > 0 and k < 400:
        samples.extend(v)
        print(f"Bucket {k}: {len(v)} sequences added to training set")
torch.save(samples, "encoded_samples.pt")

Bucket 50: 817086 sequences added to training set
Bucket 100: 172635 sequences added to training set
Bucket 150: 9048 sequences added to training set
Bucket 200: 879 sequences added to training set
Bucket 250: 202 sequences added to training set
Bucket 300: 70 sequences added to training set
Bucket 350: 34 sequences added to training set


In [10]:
samples = torch.load("encoded_samples.pt")
len(samples)

999954

In [ ]:
dataset = TrainDataset(samples)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

In [ ]:
import tqdm


EPOCHS = 3

step = 0
losses = []
model.train()
for epoch in range(EPOCHS):
    for batch, attn_masks in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch = batch.to(device)            # (B, 350) long
        attn_masks = attn_masks.to(device)  # (B, 350) bool — all True (full canvas)

        x_prob = torch.empty(1).uniform_(1e-4, 1 - 1e-4).item()

        rand_tensor = torch.rand(batch.shape, device=device)
        pred_masks = (rand_tensor < x_prob) & attn_masks

        corrupted_batch = batch.clone()
        corrupted_batch.masked_fill_(pred_masks, MASK_ID)
        corrupted_batch.masked_fill_(~attn_masks, PAD_ID)  # no-op for full-canvas case

        lm_logits = model(corrupted_batch, attn_masks)  # (B, 350, vocab_size+1)

        if pred_masks.any():
            targets = batch.clone()
            targets[targets == EOS_ID] = PAD_ID
            loss = F.cross_entropy(lm_logits[pred_masks], targets[pred_masks])
        else:
            loss = torch.tensor(0.0, device=device)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            losses.append(loss.item())
            print(f"Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, x_prob: {x_prob:.3f}")
        step += 1

Epoch 1, Step 0, Loss: 8.4856, x_prob: 0.294
Epoch 1, Step 100, Loss: 0.9562, x_prob: 0.226
Epoch 1, Step 200, Loss: 0.8724, x_prob: 0.775
Epoch 1, Step 300, Loss: 1.0098, x_prob: 0.828
Epoch 1, Step 400, Loss: 0.6920, x_prob: 0.816
Epoch 1, Step 500, Loss: 0.7582, x_prob: 0.172
Epoch 1, Step 600, Loss: 0.6457, x_prob: 0.772


KeyboardInterrupt: 